### ENTSO-R API

In [ ]:
import pandas as pd
import time
import io
import zipfile
from entsoe import EntsoePandasClient
from entsoe.files import EntsoeFileClient

### ═════════════════════════════════════════════
### SECTION 1 - CONFIGURATION
### ═════════════════════════════════════════════

In [ ]:
# Load credentials from environment variables

import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("ENTSOE_API_KEY")
USERNAME = os.getenv("ENTSOE_USERNAME")
PASSWORD = os.getenv("ENTSOE_PASSWORD")

if not API_KEY:
    raise ValueError("Missing ENTSOE_API_KEY. Create a local .env file based on .env.example.")


In [ ]:
# Norwegian bidding zones with their EIC codes

BIDDING_ZONES = {
    "NO1": "10YNO-1--------2",    # Oslo / southeast
    "NO2": "10YNO-2--------T",    # Southwest (most hydro export)
    "NO3": "10YNO-3--------J",    # Central
    "NO4": "10YNO-4--------9",    # North
    "NO5": "10Y1001A1001A48H",    # Bergen / west
}

In [ ]:
# File Library folders

FILE_FOLDERS = {
    "load":       "ActualTotalLoad_6.1.A_r3",
    "generation": "AggregatedGenerationPerType_16.1.B_C_r3",
    "reservoirs": "AggregatedFillingRateOfWaterReservoirsAndHydroStoragePlants_16.1.D_r3",
    "flows":      "PhysicalFlows_12.1.G_r3",
}

In [ ]:
# Output filenames

OUTPUT_FILES = {
    "load":       "NO_actual_load.csv",
    "generation": "NO_generation.csv",
    "reservoirs": "NO_reservoirs.csv",
    "flows":      "NO_cross_border_flows.csv",
    "prices":     "NO_day_ahead_prices.csv",
}


In [ ]:
# Time range for REST API prices

PRICE_START = pd.Timestamp("2014-01-01", tz="UTC")
PRICE_END   = pd.Timestamp("2026-01-01", tz="UTC")
BATCH_SIZE = 20    # files per ZIP download request
DELAY      = 1.0   # seconds between requests

### ═════════════════════════════════════════════
### SECTION 2 - SHARED HELPERS
### ═════════════════════════════════════════════

In [ ]:
# This function return only the rows related to Norway

def norway_filter(df):
    # Cross-border flows: keep rows where Norway is either exporter or importer
    if "OutAreaMapCode" in df.columns and "InAreaMapCode" in df.columns:
        return df[
            df["OutAreaMapCode"].str.startswith("NO", na=False) |
            df["InAreaMapCode"].str.startswith("NO", na=False)
        ]
    
    # Datasets that have a single area code column
    for col in ["AreaMapCode", "MapCode"]:
        if col in df.columns:
            return df[df[col].str.startswith("NO", na=False)]
    
    # Datasets that don't have area code but have country name
    for col in ["AreaDisplayName", "AreaName"]:
        if col in df.columns:
            return df[df[col].str.contains("Norway", case=False, na=False)]
    
    # If none of the known column names exist
    print("      ⚠️  No country column found — returning all rows")
    return df

In [ ]:
# Diagnostic function that prints a snapshot of a df after it's been saved

def print_summary(df):
    # Counts the total number of rows
    print(f"\n   Rows      : {len(df):,}")

    # Lists every column name
    print(f"   Columns   : {df.columns.tolist()}")

    # Finds the earliest and latest values
    dt_col = df.columns[0]
    print(f"   Date range: {df[dt_col].min()} → {df[dt_col].max()}")

    # Orders the area codes if exist
    if "AreaMapCode" in df.columns:
        print(f"   Areas     : {sorted(df['AreaMapCode'].unique().tolist())}")

### ═════════════════════════════════════════════
### SECTION 3 - FILE LIBRARY DOWNLOAD (load, generation, reservoirs)
### ═════════════════════════════════════════════

In [ ]:
# Uses EntsoeFileClient which authenticates via Keycloak (username + password).
# list_folder() retrieves all monthly CSV file IDs in one call (pageSize=5000).
# download_multiple_files() fetches a batch of files as a single ZIP,
# unpacks it, and returns a concatenated DataFrame automatically.
# We filter to Norway after each batch to keep memory usage low.

In [ ]:
# Downloads an entire folder from the ENTSO-E File Library in chunks,
# filters to Norway, and returns everything as a single DataFrame.


def download_file_library(client, folder, batch_size=BATCH_SIZE, delay=DELAY):
    print(f"\n   Listing files in {folder}...")

    # Discards the filenames and store just the IDs in a list
    file_dict = client.list_folder(folder)
    file_ids  = list(file_dict.values())
    total     = len(file_ids)
    print(f"   Found {total} monthly files")

    # List to store the batches
    dfs = []

    # Formulates how many batches will it need
    total_batches = (total - 1) // batch_size + 1

    # Loops through the batchs
    for i in range(0, total, batch_size):
        batch      = file_ids[i : i + batch_size]
        batch_num  = i // batch_size + 1
        print(f"   Batch {batch_num}/{total_batches} ({len(batch)} files)...", end=" ")

        # Download the files, filter to Norway and append them into a list
        try:
            df = client.download_multiple_files(batch)
            df = norway_filter(df)
            dfs.append(df)
            print(f"{len(df):,} rows")
        
        # If fails prints error
        except Exception as e:
            print(f"FAILED: {e}")
        
        # Delay between requests to avoid getting rate-limited by the server
        time.sleep(delay)

    # Stacks all the Norway-filtered batch data frames into one
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

In [ ]:
# Header
print("=" * 60)
print("ENTSO-E Norway data downloader")
print("=" * 60)

# Authentication
print("\n[1/2] FILE LIBRARY — load, generation, reservoirs")
print("─" * 60)
print("Authenticating with File Library...")

# Creates one authenticated client object to use in the following extranctions
file_client = EntsoeFileClient(username=USERNAME, pwd=PASSWORD)
print("✅ OK")

# Loops through the FILE_FOLDERS dict 
for key, folder in FILE_FOLDERS.items():
    # Prints which folders is being downloaded
    print(f"\nDownloading: {key.upper()}")
    
    # Downloads and saves the folder into csv file
    df = download_file_library(file_client, folder)
    df.to_csv(OUTPUT_FILES[key], index=False)
    
    print(f"✅ Saved → {OUTPUT_FILES[key]}")
    print_summary(df)

### ═════════════════════════════════════════════
### SECTION 4 - REST API DOWNLOAD (day-ahead prices)
### ═════════════════════════════════════════════

In [ ]:
# Uses EntsoePandasClient which authenticates via API key.
# query_day_ahead_prices() returns an hourly pandas Series.
# The library automatically splits multi-year requests into annual chunks
# to stay within the API's per-request limits.
# We loop over the 5 Norwegian bidding zones and stack them into one DataFrame.

In [ ]:
# Header
print("\n\n[2/2] REST API — day-ahead prices")
print("─" * 60)
print("Authenticating with REST API...")

# Authentication
api_client = EntsoePandasClient(api_key=API_KEY)

print("✅ OK")
print(f"\nDownloading prices for {len(BIDDING_ZONES)} bidding zones")
print(f"Period: {PRICE_START.date()} → {PRICE_END.date()}")

In [ ]:
# List to append prices
price_dfs = []

# Loops through the BIDDING_ZONES dict 
for zone_name, zone_code in BIDDING_ZONES.items():
    # Prints which zone is being downloaded
    print(f"\n   {zone_name} ({zone_code})...", end=" ")

    # Tries to call the API 
    try:
        series = api_client.query_day_ahead_prices(
            zone_code, start=PRICE_START, end=PRICE_END
        )

        # Resets index
        df = series.reset_index()

        # Renames columns
        df.columns = ["DateTime(UTC)", "Price(EUR/MWh)"]

        # Adds two columns with the zone name and code
        df["BiddingZone"]  = zone_name
        df["AreaCode"]     = zone_code

        # Appends it into the list
        price_dfs.append(df)

        # Prints the progress, showing the period
        print(f"{len(df):,} rows  ({df['DateTime(UTC)'].min().date()} → {df['DateTime(UTC)'].max().date()})")
        
        # Delay between requests to avoid getting rate-limited by the server
        time.sleep(DELAY)
    
    # If fails prints error
    except Exception as e:
        print(f"FAILED: {e}")

In [ ]:
# If price_dfs is not empty then
if price_dfs:
    # Stacks the df of the 5 zones and sorts them
    df_prices = (
        pd.concat(price_dfs, ignore_index=True)
        .sort_values(["BiddingZone", "DateTime(UTC)"])
        .reset_index(drop=True)
    )

    # Saves the df into a csv
    df_prices.to_csv(OUTPUT_FILES["prices"], index=False)

    # Describes the df
    print(f"\n✅ Saved → {OUTPUT_FILES['prices']}")
    print(f"\n   Price summary by zone (EUR/MWh):")
    print(df_prices.groupby("BiddingZone")["Price(EUR/MWh)"].describe().round(2))

### ═════════════════════════════════════════════
### SECTION 5 - FINAL SUMMARY
### ═════════════════════════════════════════════

In [ ]:
# Header
print("\n\n" + "=" * 60)
print("ALL DONE")
print("=" * 60)

# Loops through the OUTPUT_FILES
for key, path in OUTPUT_FILES.items():
    try:
        # Data frame with only the headers to count n of columns
        df = pd.read_csv(path, nrows=0)
        
        # Counts rows in files
        n  = sum(1 for _ in open(path)) - 1
        
        # Prints the information
        print(f"  {path:40s} {n:>10,} rows  {len(df.columns)} columns")
    
    # In case the file is not found
    except FileNotFoundError:
        print(f"  {path:40s} not created (check errors above)")